# Data Manipulation Using NumPy and Pandas
**Internship project | Python · Jupyter Notebook · Pandas · NumPy**

## Project objective
Load and explore a large retail dataset, identify and remove exact duplicate records, inspect and handle missing values, transform columns, perform multi-index grouping, validate the result, and export processed files.

## Dataset
Use the **Online Retail** dataset from the UCI Machine Learning Repository:
https://archive.ics.uci.edu/dataset/352/online+retail

The dataset contains retail transaction records with fields such as invoice number, product code, description, quantity, invoice date, unit price, customer ID, and country.

### Before running
1. Download the dataset from the official UCI page.
2. Extract the archive if needed.
3. Place `Online Retail.xlsx` in the same folder as this notebook.
4. Run notebook cells from top to bottom.

If your downloaded file has a different name or is stored elsewhere, update the file path in Section 2.

## 1. Install required packages (run only if needed)

In [ ]:
%pip install pandas numpy openpyxl matplotlib

## 2. Import libraries and load the dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

file_path = Path("Online Retail.xlsx")

if not file_path.exists():
    raise FileNotFoundError(
        "Place 'Online Retail.xlsx' in the same folder as this notebook, "
        "or update file_path to the correct location."
    )

df = pd.read_excel(file_path)

print("Dataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
display(df.head())

## 3. Explore the dataset

In [ ]:
# Preview records
display(df.head())
display(df.tail())

# Structure, column names, types, and memory usage
print("Column names:", df.columns.tolist())
df.info()

print("Memory used (MB):",
      round(df.memory_usage(deep=True).sum() / (1024 ** 2), 2))

# Descriptive summary
display(df.describe(include="all").T)

## 4. Standardize column names and text

In [ ]:
# Remove leading/trailing spaces from column names
df.columns = df.columns.astype(str).str.strip()

# Normalize common whitespace in text columns
for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = df[col].astype("string").str.strip()

print(df.columns.tolist())

## 5. Detect and remove exact duplicate rows

In [ ]:
duplicate_count = int(df.duplicated().sum())
print("Exact duplicate rows before cleaning:", duplicate_count)

df = df.drop_duplicates().copy()

print("Shape after removing duplicates:", df.shape)
print("Exact duplicate rows remaining:", int(df.duplicated().sum()))

## 6. Inspect missing values
Review the missing-value report before deciding how to handle each column. Missing customer IDs may be meaningful and should not be replaced with invented customer identifiers.

In [ ]:
missing_report = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percentage": (df.isna().mean() * 100).round(2)
}).sort_values("Missing_Count", ascending=False)

display(missing_report)

## 7. Handle missing values and convert data types

In [ ]:
# Convert numeric columns safely
for col in ["Quantity", "UnitPrice"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Fill missing product descriptions with a clear label
if "Description" in df.columns:
    df["Description"] = df["Description"].fillna("Unknown Product")

# Keep unknown customer IDs explicit; do not invent IDs
if "CustomerID" in df.columns:
    df["CustomerID"] = df["CustomerID"].astype("string").fillna("Unknown")

# Median imputation is used here only for missing numeric values.
# For a real analysis, confirm this is appropriate for each column.
for col in ["Quantity", "UnitPrice"]:
    if col in df.columns and df[col].isna().any():
        median_value = df[col].median()
        if pd.notna(median_value):
            df[col] = df[col].fillna(median_value)

# Convert invoice date to datetime
if "InvoiceDate" in df.columns:
    df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")

# Re-check missing values
display(df.isna().sum().to_frame("Missing_Count"))

## 8. Feature transformation with NumPy

In [ ]:
required = {"Quantity", "UnitPrice"}
missing_columns = required - set(df.columns)
if missing_columns:
    print("Cannot calculate Revenue. Missing columns:", missing_columns)
else:
    df["Revenue"] = np.where(
        df["Quantity"].notna() & df["UnitPrice"].notna(),
        df["Quantity"] * df["UnitPrice"],
        np.nan
    )

    df["Transaction_Type"] = np.select(
        [
            df["Quantity"] < 0,
            df["Quantity"] == 0,
            df["Quantity"] > 0
        ],
        [
            "Return_or_Credit",
            "Zero_Quantity",
            "Sale"
        ],
        default="Unknown"
    )

if "InvoiceDate" in df.columns:
    df["Year"] = df["InvoiceDate"].dt.year
    df["Month"] = df["InvoiceDate"].dt.month

preview_cols = [
    col for col in [
        "InvoiceNo", "StockCode", "Description", "Quantity",
        "UnitPrice", "Revenue", "Transaction_Type", "Year", "Month"
    ] if col in df.columns
]
display(df[preview_cols].head())

**Interpretation:** Revenue is calculated as quantity multiplied by unit price. Negative quantities are retained and flagged as returns/credits rather than automatically deleted. This preserves potentially important business records.

## 9. Multi-index grouping and aggregation

In [ ]:
if "Revenue" in df.columns and {"Country", "InvoiceDate", "Quantity", "InvoiceNo"}.issubset(df.columns):
    country_year_summary = df.groupby(
        ["Country", df["InvoiceDate"].dt.year.rename("Year")],
        dropna=False
    ).agg(
        Total_Revenue=("Revenue", "sum"),
        Total_Quantity=("Quantity", "sum"),
        Unique_Invoices=("InvoiceNo", "nunique")
    )
    display(country_year_summary.head(15))
    print("Index levels:", country_year_summary.index.names)
else:
    print("Grouping skipped: one or more required columns are unavailable.")

## 10. Group by country and product

In [ ]:
if "Revenue" in df.columns and {"Country", "StockCode", "Quantity", "InvoiceNo"}.issubset(df.columns):
    country_product_summary = df.groupby(
        ["Country", "StockCode"],
        dropna=False
    ).agg(
        Total_Revenue=("Revenue", "sum"),
        Total_Quantity=("Quantity", "sum"),
        Transaction_Lines=("InvoiceNo", "size")
    )
    display(country_product_summary.head(15))
else:
    print("Grouping skipped: one or more required columns are unavailable.")

## 11. Reset index for easier export

In [ ]:
if "country_year_summary" in globals():
    country_year_summary_export = country_year_summary.reset_index()
    display(country_year_summary_export.head())

## 12. Filter and sort records

In [ ]:
if "Revenue" in df.columns:
    positive_revenue = df.loc[df["Revenue"] > 0].sort_values(
        "Revenue", ascending=False
    )
    useful_cols = [
        col for col in [
            "InvoiceNo", "Description", "Quantity",
            "UnitPrice", "Revenue", "Country"
        ] if col in positive_revenue.columns
    ]
    display(positive_revenue[useful_cols].head(10))

## 13. Optional visualization

In [ ]:
if "Country" in df.columns and "Revenue" in df.columns:
    top_countries = (
        df.groupby("Country", dropna=False)["Revenue"]
          .sum()
          .sort_values(ascending=False)
          .head(10)
    )
    top_countries.plot(kind="bar", figsize=(10, 5))
    plt.title("Top 10 Countries by Total Revenue")
    plt.xlabel("Country")
    plt.ylabel("Revenue")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

## 14. Validate the cleaned dataset

In [ ]:
print("Final shape:", df.shape)
print("Exact duplicate rows remaining:", int(df.duplicated().sum()))
print("\nMissing values by column:")
display(df.isna().sum().to_frame("Missing_Count"))

if "Transaction_Type" in df.columns:
    print("\nTransaction type counts:")
    display(df["Transaction_Type"].value_counts(dropna=False))

## 15. Export cleaned data and summaries

In [ ]:
output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

df.to_csv(output_dir / "cleaned_online_retail.csv", index=False)

if "country_year_summary" in globals():
    country_year_summary.reset_index().to_csv(
        output_dir / "country_year_summary.csv", index=False
    )

if "country_product_summary" in globals():
    country_product_summary.reset_index().to_csv(
        output_dir / "country_product_summary.csv", index=False
    )

print("Export complete. Files saved in:", output_dir.resolve())

## 16. Conclusion template

After running the notebook, update this section with your actual results:

- Original dataset dimensions: **[fill in from Section 2]**
- Exact duplicate rows removed: **[fill in from Section 5]**
- Missing-value handling: **[describe the columns and methods used]**
- Features created: **Revenue, Transaction_Type, Year, Month** (where applicable)
- Grouping performed: **Country-Year** and **Country-Product**
- Exported outputs: cleaned CSV and grouped summary CSV files

Do not invent results. Use the actual output produced by your run.

## Key concepts demonstrated
- Data exploration and profiling
- Duplicate detection
- Missing-data treatment
- Data type conversion
- NumPy vectorized calculations and conditional transformations
- Pandas `groupby`, `agg`, MultiIndex, and `reset_index`
- Data validation and CSV export

## Dataset source
UCI Machine Learning Repository — Online Retail:
https://archive.ics.uci.edu/dataset/352/online+retail